# Research notebook

**This notebook is exploration only — it is not the production pipeline.**
The weekly production run is `scripts/run_weekly_pipeline.py`, scheduled via
`.github/workflows/workflow.yaml`. Everything reusable that used to be defined
inline here now lives in `src/ml_portfolio/` and is imported, not redefined, in
the cells below — this notebook is for research, walkthroughs, and one-off
analysis (factor testing, model comparison, the walk-forward backtest writeup).


# Project Set-up


In [ ]:
try: 
    import pandas as pd
    import ta
    import numpy as np
    import requests
    import sys, os
    from dotenv import load_dotenv
    from pathlib import Path
    from datetime import datetime, timedelta
    
    print("Successfully Imported all the libraries")

except Exception as e: 
    print(f"Import Error: {e}")
    raise


In [ ]:
import sys
sys.path.insert(0, str(Path('..').resolve() / 'src'))
from ml_portfolio.config import PROJECT_ROOT, DATA_DIR

load_dotenv(PROJECT_ROOT/'.env')
print(".env file successfully loaded")


# Data Extraction: 
Extract the S&P 500 constituent list from Wikipedia and historical price data via yfinance (with a stooq fallback)

### Get Ticker data

In [ ]:
from ml_portfolio.data.universe import get_sp500_universe


In [ ]:
df_holdings = get_sp500_universe()
holdings_list = df_holdings['symbol'].to_list()
print(f"{len(holdings_list)} S&P 500 symbols")

### Extract historical price data via yfinance (stooq fallback)

In [ ]:
import yfinance as yf
from pandas_datareader import data as pdr

start_date = "2021-01-01"
end_date = datetime.now().strftime("%Y-%m-%d")


In [ ]:
from ml_portfolio.data.prices import get_price_history_stooq


In [ ]:
from ml_portfolio.data.prices import get_historical_prices


In [ ]:
df = get_historical_prices(holdings_list, start_date, end_date)

In [ ]:
df.head(10)
fetched_symbols = df["symbol"].unique()
df_holdings = df_holdings[df_holdings["symbol"].isin(fetched_symbols)].reset_index(drop = True)
df_holdings.to_csv(DATA_DIR/"raw"/"holdings.csv", index = False)


### Extract SPY benchmark data (for dashboard performance metrics)
Not a model feature — used only by `app.py` to compute Sharpe/Sortino/CAGR/Volatility/Max Drawdown/Beta for the portfolio, benchmarked against SPY buy-and-hold. Reuses `get_historical_prices()` with its own `save_path` so it doesn't overwrite the S&P 500 constituent panel.

In [ ]:
df_spy = get_historical_prices(['SPY'], start_date, end_date, save_path=DATA_DIR/"raw"/"spy_price.csv")
df_spy.tail(3)

# Data Transformation

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

from ta.momentum import RSIIndicator
from ta.volatility import BollingerBands

### Import the dataset

In [ ]:
from ml_portfolio.data.io import load_data


In [ ]:
# df = read_data(DATA_DIR)
df = load_data(DATA_DIR/"raw"/"historical_price.csv")
df

### Create Target Variable
1. Calculate weekly log returns

In [ ]:
from ml_portfolio.features.target import calculate_weekly_returns


In [ ]:
df = calculate_weekly_returns(df)
df.head(3)

2. 🎯 Create Target Variable

In [ ]:
from ml_portfolio.features.target import create_target_variable


In [ ]:
df = create_target_variable(df)
df.head(3)

In [ ]:
from ml_portfolio.features.engineering import create_panel_dataset


In [ ]:
# This step could be moved to the pipeline
df = create_panel_dataset(df)
df.head(3)

In [ ]:
from ml_portfolio.features.engineering import remove_columns


In [ ]:
cols = ["open", "high", "low", "change", "changePercent", "weekly_return"]
df = remove_columns(cols, df)
df.head(3)

2. Calculate moving averages

In [ ]:
from ml_portfolio.features.engineering import calc_moving_avg


In [ ]:
from ml_portfolio.features.engineering import calc_moving_avgs


In [ ]:
windows = [200, 100, 50]
df = calc_moving_avgs(windows, df)
df.head(3)

### Risk and Volatility
(functions defined here, applied later — after the weekly filter — so the windows are genuine weeks, not trading days; see "Volatility & Skewness" below)

In [ ]:
from ml_portfolio.features.engineering import calc_volatility


In [ ]:
from ml_portfolio.features.engineering import calc_volatilties


### Short Term Reversal Factors

In [ ]:
from ml_portfolio.features.engineering import calculate_rsi


In [ ]:
from ml_portfolio.features.engineering import calculate_rsis


In [ ]:
windows = [3, 9, 14]
df = calculate_rsis(windows, df)
df.head(3)

In [ ]:
from ml_portfolio.features.engineering import calculate_bb


In [ ]:
from ml_portfolio.features.engineering import calculate_bbs


In [ ]:
bands = ["hband", "lband"]
df = calculate_bbs(bands, df)
df.head(3)

### Momentum Factor: 

In [ ]:
from ml_portfolio.features.engineering import calculate_momentum


In [ ]:
from ml_portfolio.features.engineering import calculate_momentums


In [ ]:
windows_for_momentum = [12, 6, 1]

df = calculate_momentums(windows_for_momentum, df)
df.head(3)

### Liquidity Factors

In [ ]:
# Trailing daily return (backward-looking) — distinct from weekly_log_return, which is
# deliberately forward-looking for the target. Only used as an input to Amihud
# illiquidity and market beta below, not kept as a feature itself.
df['daily_return'] = df.groupby('symbol')['close'].pct_change()
df['dollar_volume'] = df['close'] * df['volume']
df.head(3)

In [ ]:
from ml_portfolio.features.engineering import calc_amihud_illiquidity

df = calc_amihud_illiquidity(21, df)  # ~1 trading month
df.head(3)


### Market Beta

In [ ]:
from ml_portfolio.features.engineering import calc_rolling_beta

df = calc_rolling_beta(60, df)
df.head(3)  # daily_return kept for now — Fama-French loadings below also need it


### Fama-French Factor Loadings
Rolling loadings on Mkt-RF/SMB/HML/RMW/CMA, fed in as cross-sectional features (not a standalone expected-return model — see the notebook's model section for that variant).

In [ ]:
# Free, no key — Ken French's data library via pandas_datareader (same source the
# Quant Guild reference dashboard used successfully earlier this session).
ff_raw = pdr.DataReader('F-F_Research_Data_5_Factors_2x3_daily', 'famafrench', start_date, end_date)[0]
ff_factors = (ff_raw / 100).reset_index()  # library reports factors in percent
ff_factors.columns = ['date'] + list(ff_factors.columns[1:])
ff_factors['date'] = pd.to_datetime(ff_factors['date'])

ff_factor_names = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']
df = pd.merge(df, ff_factors[['date'] + ff_factor_names], on='date', how='left')
df.head(3)

In [ ]:
from ml_portfolio.features.engineering import calc_ff_loadings

df = calc_ff_loadings(60, df, ff_factor_names)
df = df.drop(columns=['daily_return'] + ff_factor_names)  # raw per-date factor values don't vary cross-sectionally; only the loadings do
df.head(3)


### Filter Datasets:

In [ ]:
from ml_portfolio.features.engineering import wed_thurs_selector


In [ ]:
from ml_portfolio.features.engineering import filter_data


In [ ]:
df = filter_data(df)
df.head(3)

### Volatility & Skewness (weekly-window corrected)
Computed here, after the Wed/Thu filter, so a window of 4/26/52 genuinely means 4/26/52 *weeks* — previously this ran on the daily panel, so the same window numbers meant 4/26/52 *trading days* instead.

In [ ]:
volatility_dict = {
    'vol_1M': 4,    # 4 weeks
    'vol_6M': 26,   # 26 weeks
    'vol_12M': 52   # 52 weeks
}

df = calc_volatilties(volatility_dict, df)
df.head(3)

In [ ]:
from ml_portfolio.features.engineering import calc_skewness, calc_skewnesses


In [ ]:
skewness_dict = {
    'skew_1M': 4,
    'skew_6M': 26,
    'skew_12M': 52
}

df = calc_skewnesses(skewness_dict, df)
df.head(3)

### Sector-Relative Factors

In [ ]:
# Each stock's RSI_14 / momentum_6M minus its own GICS sector's average that week —
# isolates stock-specific signal from sector-wide moves.
df_sectors = load_data(DATA_DIR/"raw"/"holdings.csv")[['symbol', 'gics_sector']]
df = pd.merge(df, df_sectors, on='symbol', how='left')

df['RSI_14_sector_relative'] = df['RSI_14'] - df.groupby(['date', 'gics_sector'])['RSI_14'].transform('mean')
df['momentum_6M_sector_relative'] = df['momentum_6M'] - df.groupby(['date', 'gics_sector'])['momentum_6M'].transform('mean')

df = df.drop(columns=['gics_sector'])  # categorical, not a model input itself
df.head(3)

In [ ]:
# beta_60d needs a full 60-row warmup (min_periods=window) and skewness needs at least
# 3 points to be defined — both reintroduce NaNs after filter_data()'s own dropna already
# ran, so clean up again before saving/modeling.
df = df.dropna(ignore_index=True)
df.head(3)

In [ ]:
from ml_portfolio.features.engineering import save_processed_data


In [ ]:
save_processed_data(df, DATA_DIR)

### Create Custom transformer
E.g. Remove/Fill in Nan values, log transformation, standardization

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import BaseEstimator, TransformerMixin
import pandas as pd

In [ ]:
from ml_portfolio.models.pipeline import LogTransformer


### 💽Create training and testing variable

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.model_selection import TimeSeriesSplit
from sklearn.model_selection import KFold

import joblib

In [ ]:
from ml_portfolio.models.train import create_variables

X, y = create_variables(df)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2)

In [ ]:
from ml_portfolio.models.train import time_aware_split


In [ ]:
X_train, X_test, y_train, y_test = time_aware_split(df)

In [ ]:
# tscv = TimeSeriesSplit(n_splits=3)
# for i, (train_index, test_index) in enumerate(tscv.split(df)):
#     print(f"Fold {i + 1}:")
#     print(f"  Train: index={train_index}")
#     print(f"  Test:  index={test_index}")

### 🏭 Build a pipeline to transform features

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

In [ ]:
#build pipeline
log_transform_features = ['MA_200', 'MA_100', 'MA_50', 'hband', 'lband']
# cols_to_drop = ['Ticker', 'date', 'weekly_log_return']

cols_to_drop = ['symbol', 'date', 'weekly_log_return']

# Example: Drop columns 'col1' and 'col2'
col_dropper = ColumnTransformer(
    transformers=[
        ('drop_cols', 'drop', cols_to_drop)
    ],
    remainder='passthrough'  # keeps all other columns
)

ridge_pipeline = Pipeline(steps = [ 
    ('log_transformers', LogTransformer(log_transform_features)),
    ('col_dropper', col_dropper), 
    ('scaler', StandardScaler()), 
    ('regressor', Ridge())
])

In [ ]:
ridge_model = ridge_pipeline.fit(X_train, y_train)

In [ ]:
filename = "ridge_regression_model.sav"
joblib.dump(ridge_model, open(filename, 'wb'))

In [ ]:
ridge_model = joblib.load(open(filename, 'rb'))
y_pred = ridge_model.predict(X_test)
print(y_pred)

In [ ]:
from scipy.stats import spearmanr

r_squared = ridge_model.score(X_test, y_test)
ic, ic_pvalue = spearmanr(y_pred, y_test)

df_eval = pd.DataFrame({'y_pred': y_pred, 'y_test': y_test.values})
df_eval['decile'] = pd.qcut(df_eval['y_pred'], 10, labels = False, duplicates = 'drop')
decile_returns = df_eval.groupby('decile')['y_test'].mean()
decile_spread = decile_returns.iloc[-1] - decile_returns.iloc[0]

print(f"R²: {r_squared:.4f}")
print(f"Spearman IC: {ic:.4f} (p={ic_pvalue:.4f})")
print(f"Top-Bottom Decile Spread: {decile_spread:.4f}")

# Factor Testing
Evaluate the 8 new candidate factors (liquidity, Amihud, beta, skewness, sector-relative RSI/momentum) alongside the 17 existing ones, using the same held-out test set as the main evaluation above.

In [ ]:
from ml_portfolio.models.evaluate import univariate_ic, redundancy_check, ablation_test, sub_period_ic


In [ ]:
baseline_cols = ['close', 'volume', 'vwap', 'MA_200', 'MA_100', 'MA_50', 'vol_1M', 'vol_6M', 'vol_12M',
                 'RSI_3', 'RSI_9', 'RSI_14', 'hband', 'lband', 'momentum_12M', 'momentum_6M', 'momentum_1M']
new_factor_cols = ['dollar_volume', 'amihud_illiquidity', 'beta_60d', 'skew_1M', 'skew_6M', 'skew_12M',
                    'RSI_14_sector_relative', 'momentum_6M_sector_relative']

test_df = X_test.assign(target = y_test)
baseline_result = ablation_test(X_train, X_test, y_train, y_test, baseline_cols)
print(f"Baseline (17 existing factors): IC={baseline_result['ic']:.4f}, decile_spread={baseline_result['decile_spread']:.4f}")

results = []

for factor in baseline_cols:
    ic, ic_p = univariate_ic(test_df, factor)
    reduced_cols = [c for c in baseline_cols if c != factor]
    ablated = ablation_test(X_train, X_test, y_train, y_test, reduced_cols)
    sub_ics = sub_period_ic(test_df, factor)
    results.append({
        'factor': factor, 'type': 'existing',
        'univariate_ic': ic, 'ic_pvalue': ic_p,
        'max_redundancy_corr': None,
        'ablation_delta_ic': baseline_result['ic'] - ablated['ic'],
        'ablation_delta_decile_spread': baseline_result['decile_spread'] - ablated['decile_spread'],
        'subperiod_ic_1': sub_ics[0], 'subperiod_ic_2': sub_ics[1],
    })

for factor in new_factor_cols:
    ic, ic_p = univariate_ic(test_df, factor)
    redundant = redundancy_check(test_df, factor, baseline_cols)
    expanded_cols = baseline_cols + [factor]
    added = ablation_test(X_train, X_test, y_train, y_test, expanded_cols)
    sub_ics = sub_period_ic(test_df, factor)
    results.append({
        'factor': factor, 'type': 'new',
        'univariate_ic': ic, 'ic_pvalue': ic_p,
        'max_redundancy_corr': redundant.abs().max() if len(redundant) else 0.0,
        'ablation_delta_ic': added['ic'] - baseline_result['ic'],
        'ablation_delta_decile_spread': added['decile_spread'] - baseline_result['decile_spread'],
        'subperiod_ic_1': sub_ics[0], 'subperiod_ic_2': sub_ics[1],
    })

summary_table = pd.DataFrame(results)
summary_table

# Additional Return-Prediction Models
Ridge (above) plus three more, tested the same way, then combined by IC-weighted average.

### Fama-French Standalone Signal
Not fit to our target at all — a stock's own factor loadings combined with each factor's historical average return, in the classic asset-pricing "expected return" formula. Built specifically as an ensemble diversification candidate, not expected to be a strong standalone predictor on its own.

In [ ]:
from ml_portfolio.models.pipeline import fama_french_standalone_signal

# Evaluated after the validation split below, so the historical mean factor return
# comes strictly from the training slice, consistent with how the other models are
# trained (see the note there on why this matters).


### Validation Split
Carved out of the existing `X_train`/`y_train` only — `X_test`/`y_test` stay exactly as they were for Part 4, so results stay comparable. Needed so IC-weighted combination weights come from data the final test-set evaluation never touches.

In [ ]:
from ml_portfolio.models.evaluate import carve_validation_split

X_train_final, X_val, y_train_final, y_val = carve_validation_split(X_train, y_train)
print(f"train: {len(X_train_final)}, validation: {len(X_val)}, test: {len(X_test)}")


### Model Pipelines
Ridge, ElasticNet, and RandomForest — same feature set (everything currently in `X_train`, i.e. exactly what the main Ridge model above already uses: the 17 original factors + Part 4's new ones + this section's Fama-French loadings), same pipeline shape, only the final regressor differs.

In [ ]:
from sklearn.linear_model import ElasticNetCV
from sklearn.ensemble import RandomForestRegressor

from ml_portfolio.models.pipeline import build_model_pipeline

all_feature_cols = [c for c in X_train.columns if c not in ['symbol', 'date', 'weekly_log_return']]

# Note: the original ridge_model (main evaluation section above) was fit on the full
# X_train, which now includes what carve_validation_split just held out as X_val —
# using it here would let Ridge "see" validation data that ElasticNet/RandomForest
# never do, biasing the IC-weighted combination in Ridge's favor. Refit a version
# specifically on X_train_final so all four models are compared on equal footing.
ridge_pipeline_ensemble = build_model_pipeline(all_feature_cols, Ridge())
ridge_model_ensemble = ridge_pipeline_ensemble.fit(X_train_final, y_train_final)

# Plain ElasticNet(alpha=1.0) zeroed out every feature on this weak-signal, many-
# feature dataset — predicted the same constant value for every row. ElasticNetCV
# searches alpha (and l1_ratio) via internal cross-validation on the training data
# only, so it doesn't over-regularize into a degenerate constant model.
elasticnet_pipeline = build_model_pipeline(all_feature_cols, ElasticNetCV(l1_ratio=[.1, .5, .7, .9, .95, .99, 1], cv=5, n_jobs=-1))
elasticnet_model = elasticnet_pipeline.fit(X_train_final, y_train_final)

rf_pipeline = build_model_pipeline(all_feature_cols, RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42, n_jobs=-1))
rf_model = rf_pipeline.fit(X_train_final, y_train_final)

print("Ridge (ensemble version), ElasticNet, and RandomForest fit")


In [ ]:
# Historical mean factor return from the training slice only (X_train_final, not the
# original X_train which now includes what's held out as X_val) — using validation-
# or test-period factor performance here would be look-ahead.
mean_factor_returns = ff_factors[ff_factors['date'].isin(X_train_final['date'])][ff_factor_names].mean()
ff_standalone_pred_test = fama_french_standalone_signal(X_test, mean_factor_returns)
ic, ic_p = spearmanr(ff_standalone_pred_test, y_test)
print(f"Fama-French standalone: test IC={ic:.4f} (p={ic_p:.4f})")

# Model Testing
Parallel to the Factor Testing section above, applied to model predictions instead of raw factors: out-of-sample IC/decile-spread/sub-period-stability per model, and a pairwise correlation matrix — the model-level equivalent of the factor redundancy check.

In [ ]:
from ml_portfolio.models.evaluate import evaluate_predictions, sub_period_ic_predictions

model_predictions_test = {
    'Ridge': ridge_model_ensemble.predict(X_test),
    'ElasticNet': elasticnet_model.predict(X_test),
    'RandomForest': rf_model.predict(X_test),
    'FamaFrench_standalone': ff_standalone_pred_test,
}

model_results = []
for name, preds in model_predictions_test.items():
    metrics = evaluate_predictions(preds, y_test)
    sub_ics = sub_period_ic_predictions(X_test['date'], preds, y_test)
    model_results.append({
        'model': name,
        'ic': metrics['ic'], 'ic_pvalue': metrics['ic_pvalue'], 'decile_spread': metrics['decile_spread'],
        'subperiod_ic_1': sub_ics[0], 'subperiod_ic_2': sub_ics[1],
    })

model_summary = pd.DataFrame(model_results)
print(model_summary.to_string(index=False))


In [ ]:
# Pairwise correlation between models' predictions — the model-level equivalent of
# the factor redundancy check. Low correlation between two models signals real
# diversification value for the combination step below; high correlation means one
# is redundant with the other regardless of its own IC.
model_correlation_matrix = pd.DataFrame(model_predictions_test).corr()
model_correlation_matrix

# IC-Weighted Signal Combination
Weights come from the validation set only — never the test set — so the final test-set IC is an honest measure of whether combining actually helped, not a combination quietly fit to the same data it's evaluated on.

In [ ]:
from ml_portfolio.models.evaluate import _floored_ic, standardize

model_predictions_val = {
    'Ridge': ridge_model_ensemble.predict(X_val),
    'ElasticNet': elasticnet_model.predict(X_val),
    'RandomForest': rf_model.predict(X_val),
    'FamaFrench_standalone': fama_french_standalone_signal(X_val, mean_factor_returns),
}

# IC-weights: negative or undefined (e.g. a degenerate constant-output model) IC is
# floored at zero — excluded from the blend, not inverted or allowed to error out.
validation_ics = {name: _floored_ic(preds, y_val) for name, preds in model_predictions_val.items()}
total_ic = sum(validation_ics.values())
weights = ({name: ic / total_ic for name, ic in validation_ics.items()} if total_ic > 0
           else {name: 1 / len(validation_ics) for name in validation_ics})

print("Validation IC per model:", {k: round(v, 4) for k, v in validation_ics.items()})
print("Combination weights:", {k: round(v, 4) for k, v in weights.items()})

# Standardize each model's predictions using validation-set mean/std (not test-set),
# so different models' output scales don't distort the weighted sum.
standardization_stats = {name: (preds.mean(), preds.std()) for name, preds in model_predictions_val.items()}


In [ ]:
combined_test_pred = np.zeros(len(y_test))
for name, preds in model_predictions_test.items():
    combined_test_pred += weights[name] * standardize(preds, standardization_stats[name])

combined_metrics = evaluate_predictions(combined_test_pred, y_test)

comparison = pd.concat([
    model_summary[['model', 'ic', 'decile_spread']],
    pd.DataFrame([{'model': 'Combined (IC-weighted)', 'ic': combined_metrics['ic'], 'decile_spread': combined_metrics['decile_spread']}])
], ignore_index=True)

print("Test-set comparison — does the combination actually beat every individual model?\n")
print(comparison.to_string(index=False))

# Create Weekly Stock Portfolio

In [ ]:
from ml_portfolio.data.io import get_last_week_data


In [ ]:
filename = DATA_DIR/'processed'/'processed_historical_price.csv'
df_last_week = get_last_week_data(filename).reset_index(drop = True)
X_last_week = df_last_week.drop(columns = ['target'])
y_last_week = df_last_week['target']


In [ ]:
y_pred = ridge_model.predict(X_last_week)
df_pred = pd.DataFrame(y_pred, columns = ['predicted_return'])

df_pred

In [ ]:
from ml_portfolio.portfolio.construction import create_weekly_stock_portfolio


In [ ]:
df_stock_portfolio = create_weekly_stock_portfolio(df_last_week['symbol'], df_pred)
df_stock_portfolio

In [ ]:
from ml_portfolio.portfolio.tracking import update_stock_portfolio


In [ ]:
df_historical_price = load_data(DATA_DIR/'raw'/'historical_price.csv')
df_weekly_portfolio = update_stock_portfolio(df_stock_portfolio, df_historical_price, df_last_week['date'].iloc[0])
# [1 / df_weekly_portfolio['symbol'].nunique() ] 
df_weekly_portfolio

In [ ]:
from ml_portfolio.portfolio.tracking import calculate_portfolio_metrics


In [ ]:
df_weekly_perf = calculate_portfolio_metrics(df_weekly_portfolio)
df_weekly_perf


In [ ]:
from ml_portfolio.portfolio.tracking import historical_performance


In [ ]:
hist_perf_file_path = DATA_DIR/'processed'/'historical_performance.csv'
# Anchor to df_last_week's own date -- the exact same date that already drives the
# stock selection and pricing above -- not wall-clock datetime.now() (same staleness
# fix as get_last_week_data() and update_stock_portfolio()) and not df_weekly_perf's
# own max date either (that can land a day or two later within the same calendar
# week, since df_weekly_perf has one row per day of the week's raw price data, which
# would needlessly create a near-duplicate row next to the walk-forward backtest's
# Wed/Thu-anchored dates for what's actually the same week).
prev_date = df_last_week['date'].iloc[0]

df_hist_perf = historical_performance(hist_perf_file_path, df_weekly_perf, prev_date)


df_hist_perf

# Walk-Forward Backtest — Fill the Historical Performance Gap
`historical_performance.csv` has a gap between 2026-01-16 and 2026-07-24: this
return-prediction pipeline didn't exist yet in January, and the CI workflow that
*did* run during that window had no step to persist its output (fixed separately —
see the workflow file). That data was never computed anywhere, so it can't be
recovered — instead, this section **fills the gap with a walk-forward backtest**
of the current strategy: for each historical week, refit Ridge on strictly-prior
data only (expanding window, no look-ahead), predict, take the top 20 by
predicted return, and record the realized equal-weight return.

Only 2026-01-21 through 2026-05-27 can be filled this way. Ken French's
Fama-French data library — one of the model's feature sources — lags real time
by roughly two months, so weeks after that don't have a complete, consistent
feature set for anyone right now, not just for this backtest. That remaining
~8-week stretch stays an honest, documented gap rather than being filled with a
different, inconsistent reduced-feature model.

**This is a simulation, not a recovered historical record** — no real capital
was tracking this strategy during the gap. Rows produced here are marked
`source='backtest'` in `historical_performance.csv`, distinct from
`source='live'` for genuinely tracked weeks. Treat the resulting performance
numbers with real skepticism: per-week IC over this window ranged from -0.19 to
+0.36 (mean 0.03, consistent with this project's other Ridge results) — the
backtest's high average return reflects a handful of favorable weeks within a
short, noisy 19-week sample, not a validated, repeatable edge.

In [ ]:
from ml_portfolio.backtest.walk_forward import backfill_backtest_gap


In [ ]:
# Idempotent: skips weeks already present in historical_performance.csv, so this
# is a no-op once the gap is filled. Safe to re-run on every scheduled CI trigger
# (this is exactly what scripts/run_weekly_pipeline.py also calls in production).
hist_perf_file_path = DATA_DIR / 'processed' / 'historical_performance.csv'
backfill_backtest_gap(DATA_DIR / 'processed' / 'processed_historical_price.csv', hist_perf_file_path)

combined = pd.read_csv(hist_perf_file_path)
print(f"historical_performance.csv: {len(combined)} rows ({(combined['source'] == 'backtest').sum()} backtested)")
combined
